In [2]:
import sqlite3
import random
import time
import requests
import pandas as pd
from datetime import date, timedelta

random.seed(7)

conn = sqlite3.connect("retail.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders")
cur.execute("DROP TABLE IF EXISTS customers")

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    segment TEXT
)
""")
cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cities = ["Seoul", "Busan", "Incheon", "Daegu", "Gwangju"]
segments = ["Retail", "Wholesale", "Online"]
names = ["Kim Minji","Park Jisoo","Lee Jaehyun","Choi Seoyeon","Jung Hyunwoo",
         "Kang Yuna","Yoon Doyoon","Han Soomin","Oh Jiwoo","Shin Taeyang",
         "Baek Narae","Song Junho","Moon Hana","Ahn Jiho","Jang Yerin"]

customers = [(i, name, random.choice(cities), random.choice(segments))
             for i, name in enumerate(names, start=1)]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

start_date = date(2026, 1, 1)
order_id, orders = 1, []
for c in customers:
    for _ in range(random.randint(2, 8)):
        d = start_date + timedelta(days=random.randint(0, 270))
        amt = round(random.uniform(15000, 450000), -2)
        orders.append((order_id, c[0], d.isoformat(), amt))
        order_id += 1
cur.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
conn.commit()

print(f"retail.db ready: {len(customers)} customers, {len(orders)} orders.")

retail.db ready: 15 customers, 73 orders.


In [3]:
def customers_in_city_segment(conn, city, segment):
    """
    conn: sqlite3 connection
    city: str, e.g. "Seoul"
    segment: str, e.g. "Online"
    returns: pandas DataFrame with columns name, city, segment
    """
    query = """
        SELECT name, city, segment
        FROM customers
        WHERE city = ? AND segment = ?
    """
    return pd.read_sql(query, conn, params=(city, segment))

# Test
print(customers_in_city_segment(conn, "Seoul", "Online"))

         name   city segment
0   Kang Yuna  Seoul  Online
1  Song Junho  Seoul  Online


In [4]:
def orders_with_customer_names(conn):
    """
    returns: pandas DataFrame with columns name, order_date, amount (first 10 rows)
    """
    query = """
        SELECT c.name, o.order_date, o.amount
        FROM orders AS o
        INNER JOIN customers AS c ON o.customer_id = c.customer_id
        LIMIT 10
    """
    return pd.read_sql(query, conn)

print(orders_with_customer_names(conn))

         name  order_date    amount
0   Kim Minji  2026-01-26  439700.0
1   Kim Minji  2026-01-24  257100.0
2   Kim Minji  2026-03-10  141000.0
3   Kim Minji  2026-03-15  250200.0
4   Kim Minji  2026-06-07  258700.0
5  Park Jisoo  2026-04-03   59800.0
6  Park Jisoo  2026-04-07  177000.0
7  Park Jisoo  2026-02-02  260500.0
8  Park Jisoo  2026-04-16  230900.0
9  Park Jisoo  2026-08-07  353100.0


In [5]:
def high_spending_customers(conn, min_total=300000):
    """
    returns: pandas DataFrame with columns name, total_spent
    """
    query = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers AS c
        INNER JOIN orders AS o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        HAVING SUM(o.amount) > ?
        ORDER BY total_spent DESC
    """
    return pd.read_sql(query, conn, params=(min_total,))

print(high_spending_customers(conn))

            name  total_spent
0     Jang Yerin    2167100.0
1    Yoon Doyoon    1609600.0
2     Baek Narae    1559500.0
3     Park Jisoo    1523300.0
4       Oh Jiwoo    1437300.0
5     Song Junho    1359000.0
6      Kim Minji    1346700.0
7       Ahn Jiho    1129700.0
8      Moon Hana     895700.0
9   Jung Hyunwoo     861200.0
10  Shin Taeyang     619200.0
11   Lee Jaehyun     613200.0
12    Han Soomin     427900.0
13  Choi Seoyeon     374600.0


In [6]:
def top_n_customers(conn, n=3):
    """
    returns: pandas DataFrame with columns name, total_spent — exactly n rows
    """
    query = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers AS c
        INNER JOIN orders AS o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        ORDER BY total_spent DESC
        LIMIT ?
    """
    return pd.read_sql(query, conn, params=(n,))

print(top_n_customers(conn, 3))

          name  total_spent
0   Jang Yerin    2167100.0
1  Yoon Doyoon    1609600.0
2   Baek Narae    1559500.0


In [7]:
def find_orders_by_customer(conn, customer_name):
    """
    conn: sqlite3 connection
    customer_name: str
    returns: list of tuples (order_id, order_date, amount)
    """
    sql = """
        SELECT o.order_id, o.order_date, o.amount
        FROM orders AS o
        INNER JOIN customers AS c ON o.customer_id = c.customer_id
        WHERE c.name = ?
        ORDER BY o.order_date
    """
    cursor = conn.cursor()
    cursor.execute(sql, (customer_name,))
    return cursor.fetchall()

print(find_orders_by_customer(conn, "Kim Minji"))

[(2, '2026-01-24', 257100.0), (1, '2026-01-26', 439700.0), (3, '2026-03-10', 141000.0), (4, '2026-03-15', 250200.0), (5, '2026-06-07', 258700.0)]
